# 分钟风险信息：固定年度时间外预测
日线1/5/22日平方收益历史 vs 增加同尺度分钟实现方差，固定Ridge10、训练残差smearing。三模型×三年9拟合；另有直接日线22日历史方差。未知分钟预测回退配对日线，不删除压力日。不改风险预算或交易规则。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from src.workflow.minute_action_value import _source
from src.workflow.minute_pit_allocator import _dataset
from src.objective.relative_wealth import stationary_block_bootstrap_indices
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/minute_risk_oof_20261015';p=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
training,ta=_source(root,p['training_source'],['inputs/daily_frame','inputs/native_prices','inputs/minute_features','inputs/timing',*[f'models/base_{year}_71' for year in p['years']]])
prices,pa=_source(root,p['price_source'],['inputs/frame']);base=training['inputs/daily_frame'];index=base.index;marks=prices['inputs/frame'].set_index('signal_date').reindex(index);minute=training['inputs/minute_features'].reindex(index)
close=training['inputs/native_prices'].xs('SOXX',level='instrument')['$close'];daily_sq=np.log(close/close.shift(1)).pow(2).reindex(index)
# Source field stores log standard deviation, not log variance.
rv=np.exp(2*minute.minute_soxx_log_rv5).where(minute.minute_core_ready.eq(1))
feature=pd.DataFrame(index=index);daily_names=[];minute_names=[]
for window in p['windows']:
 daily_names.append('daily_'+str(window));minute_names.append('minute_'+str(window));feature[daily_names[-1]]=np.log(daily_sq.rolling(window,min_periods=window).mean().clip(lower=p['log_floor']));feature[minute_names[-1]]=np.log(rv.rolling(window,min_periods=window).mean().clip(lower=p['log_floor']))
known=feature[minute_names].notna().all(axis=1);decisions=training['inputs/timing'].decision_at
available=pd.to_datetime(minute.minute_feature_available_at);known=known&decisions.notna();feature.loc[~known,minute_names]=np.nan;assert (available[known]<=decisions[known]).all()
y=np.log(marks.soxx_exit_open/marks.soxx_open).pow(2);maturity=pd.to_datetime(marks.exit_date).dt.tz_localize('America/New_York')+pd.Timedelta(hours=9,minutes=49)
meta=pd.DataFrame({'label_available_at':maturity,'minute_known':known,'trade_date':marks.trade_date},index=index)
datasets={};domains={}
for year in p['years']:
 original=training[f'models/base_{year}_71'];dates=index[pd.to_datetime(marks.trade_date).dt.year.eq(year)];domains[year]=dates
 datasets[year]=_dataset(feature,pd.DataFrame({'risk':y}),pd.DataFrame(index=index),meta,{'train':('2011-01-03',str(original.train_end.date())),'predict':(str(dates.min().date()),str(dates.max().date()))})
models={};audits=[];parts=[]
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=p,source_audits={'training':ta,'prices':pa},features=feature,labels=pd.DataFrame({'risk':y}),metadata=meta);rec.log_params(maximum_new_fits=9,new_backtests=0)
 for year in p['years']:
  ds=datasets[year];a=ds.prepare('train',col_set=['feature','label','meta']);out=ds.prepare('predict',col_set=['feature','label','meta']);cutoff=pd.Timestamp(f'{year}-01-01',tz='America/New_York')
  eligible=pd.to_datetime(a[('meta','label_available_at')]).lt(cutoff)&np.isfinite(a[('label','risk')])&np.isfinite(a['feature'][daily_names]).all(axis=1);paired=eligible&a[('meta','minute_known')].astype(bool)
  prediction=pd.DataFrame(index=domains[year]);prediction['risk']=out[('label','risk')].to_numpy();prediction['year']=year;prediction['minute_known']=out[('meta','minute_known')].to_numpy(bool);prediction['label_available_at']=out[('meta','label_available_at')].to_numpy()
  for family,columns,mask in [('daily_matched',daily_names,paired),('daily_all',daily_names,eligible),('minute',daily_names+minute_names,paired)]:
   train=a.loc[mask];assert len(train)>=p['minimum_rows'];x=train['feature'][columns];target=np.log(train[('label','risk')].to_numpy()+p['log_floor']);model=make_pipeline(StandardScaler(),Ridge(alpha=p['ridge_alpha']));model.fit(x,target);residual=target-model.predict(x);smear=float(np.exp(residual).mean());assert np.isfinite(smear) and smear>0
   values=out['feature'][columns];valid=np.isfinite(values).all(axis=1);forecast=np.full(len(out),np.nan);forecast[valid]=np.exp(model.predict(values.loc[valid]))*smear;assert np.isfinite(forecast[valid]).all() and (forecast[valid]>0).all()
   prediction[family]=forecast
   if family=='minute':prediction.loc[~valid.to_numpy(),'minute']=prediction.loc[~valid.to_numpy(),'daily_matched']
   models[f'{year}_{family}']={'model':model,'smearing_factor':smear,'features':columns};audits.append({'year':year,'family':family,'training_rows':len(train),'last_label_available_at':str(train[('meta','label_available_at')].max()),'cutoff':str(cutoff),'smearing_factor':smear,'minute_unknown_prediction_rows':int((~prediction.minute_known).sum())});rec.log_metrics(completed_fits=len(models))
  prediction['daily22']=np.exp(out[('feature','daily_22')].to_numpy());parts.append(prediction)
 forecast=pd.concat(parts).sort_index();assert len(forecast)==756 and not forecast.index.duplicated().any();assert np.isfinite(forecast[['minute','daily_matched','daily_all','daily22']]).all().all();rec.save_objects(artifact_path='models',fitted=models);rec.save_objects(artifact_path='analysis',forecasts=forecast,training=audits)
valid=pd.to_datetime(forecast.label_available_at).lt(pd.Timestamp(p['cutoff']))&np.isfinite(forecast.risk);loss=pd.DataFrame(index=forecast.index)
for family in ['minute','daily_matched','daily_all','daily22']:
 h=forecast[family];loss[family]=(np.log(h)+forecast.risk/h).where(valid)
rows=[];ci=[];indices=stationary_block_bootstrap_indices(observation_count=len(forecast),path_length=len(forecast),path_count=p['bootstrap_resamples'],expected_block_length=p['bootstrap_mean_block'],seed=p['bootstrap_seed'])
for reference in ['daily_matched','daily_all','daily22']:
 improvement=loss[reference]-loss.minute
 for year,mask in [('all',np.ones(len(forecast),bool)),*[(str(year),forecast.year.eq(year).to_numpy()) for year in p['years']]]:
  z=improvement.loc[mask];rows.append({'reference':reference,'year':year,'known_rows':int(z.notna().sum()),'mean_qlike_improvement':float(z.mean()),'minute_fallback_rows':int((~forecast.minute_known&pd.Series(mask,index=forecast.index)).sum())})
 boot=np.nanmean(improvement.to_numpy()[indices],axis=1);lower,upper=np.quantile(boot,[p['lower_tail_probability'],1-p['lower_tail_probability']]);ci.append({'reference':reference,'mean_improvement':improvement.mean(),'lower':lower,'upper':upper,'lower_positive':bool(lower>0)})
table=pd.DataFrame(rows);intervals=pd.DataFrame(ci);annual_positive=bool(table[table.year.ne('all')].mean_qlike_improvement.gt(0).all());passed=bool(intervals.lower_positive.all() and annual_positive)
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','models/*','analysis/forecasts','analysis/forecast_losses','analysis/bootstrap_indices','analysis/training']}
summary={'experiment':card.name,'decision':'risk_information_pass_needs_account_validation' if passed else 'not_promoted_minute_risk_information_failed','new_fits':len(models),'new_backtests':0,'known_evaluation_rows':int(valid.sum()),'all_comparator_lower_bounds_positive':bool(intervals.lower_positive.all()),'all_year_comparator_improvements_positive':annual_positive,'provider_uri':str(provider),'region':'us','as_of':'2026-10-09','recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'objective_achieved':False,'scope':p['scope']}
rec.save_objects(artifact_path='analysis',forecast_losses=loss,bootstrap_indices=indices,comparison=table,intervals=intervals,summary=summary);rec.log_metrics(registered_artifact_count=13)
for name,value in [('forecast_comparison',table),('confidence_intervals',intervals),('training_audit',pd.DataFrame(audits))]:value.to_csv(card/(name+'.csv'),index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}))
binding=json.loads((root/'research/opening_state_training_20261015/nas_binding.json').read_text());binding.pop('evidence',None);(card/'nas_binding.json').write_text(json.dumps(binding,indent=2))
a=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},indent=2))
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(table.to_string(index=False));print(intervals.to_string(index=False))


# 分钟风险预测增量：9次训练完成，信息门未通过

本轮检验不同于此前H1方向配置和旧日线波动路线：加入前一完整交易日的分钟实现方差信息，预测SOXX实际0947至下一0947持有区间的对数收益平方。它是条件二阶矩的高噪声代理，不等于已观测真实条件方差，也不测量持有期最大回撤。方法借鉴HAR的日/周/月尺度思想，不声称复制论文或其收益。来源：https://search.usi.ch/en/publications/1448/a-simple-approximate-long-memory-model-of-realized-volatility 。

固定2020/2021/2022年度时间外预测，沿用原前一年12月2日训练截止；每年拟合三模型，共9拟合、0新交易回测：日线同样本、日线全部可用样本、日线加分钟候选。另保留直接22日历史平方收益均值，不需拟合。所有拟合固定Ridge alpha=10、训练内标准化、对数目标和训练残差smearing修正，无超参搜索。

日线输入是收盘对数收益平方的1/5/22日均值；候选增加五分钟收益累计实现方差的同尺度均值。源minute_soxx_log_rv5实际存log标准差，已用exp(2×字段)还原方差，未误当log方差。日志零值下限1e-12预先固定，不是按成绩调整。所有输入均在下一决策前可用，目标只有结束成交后0949才成熟。

候选/配对日线训练样本数为265/496/748，全部可用日线为286/539/791。分钟异常及22日完整窗口产生的缺失保留为NaN；预测时22个未知日回退配对日线，完整756日历保留，755日目标在2023-01-01前成熟。没有删除2020压力日。初次执行因完整源最后一日无下一决策时间，在0拟合时拦截；修正为无决策行不可用，实际决策日仍严格核验，科学计划未改。

QLIKE使用log(预测二阶矩)+实际平方收益/预测二阶矩；以下为对照损失减候选损失，正数代表候选较好，不是收益率或百分比：

| 对照 | 2020改善 | 2021改善 | 2022改善 | 全期改善 | 区块端点 |
|---|---:|---:|---:|---:|---:|
| 同样本日线拟合 | -0.03279 | +0.07459 | +0.05992 | +0.03375 | [-0.01834,+0.07992] |
| 全可用样本日线拟合 | -0.04619 | +0.07332 | +0.03553 | +0.02076 | [-0.03178,+0.06695] |
| 直接22日历史二阶矩 | +0.00069 | +0.10742 | +0.00435 | +0.03752 | [-0.02784,+0.09760] |

固定均长63日平稳区块2000次；端点为1.667%和98.333%分位，下界对三对照作单侧Bonferroni校正。全部下界不为正，候选还在2020年弱于两个日线拟合对照，预设信息门失败。区间仅条件于这批历史预测，不含重新拟合与既有研究选择的不确定性。

决策not_promoted_minute_risk_information_failed。固定风险特征路线关闭，不换时间尺度、Ridge强度、标签或缺失规则挽救，不接入仓位、不提高风险预算、不声称分钟风险预测已经有效。下一项真正的信息需要是更长且时序合格的分钟风险训练/验证证据，或有独立机制的风险输入；265条首折训练及高噪声平方收益限制必须保留，不能解释为单一异常日期导致全部失败。

无新的收益改善，生产不变，稳定净SOXX超额未完成。可复现notebook、模型、逐日预测和QLIKE损失在Qlib Recorder；NAS冷恢复逐对象哈希核对通过。
